### 生成模型（二）
<div align="center">
  <img src="class_images/taxonomy_GM.jpg" width="500">
</div>

lec_13中介绍的生成模型是给出输出的具体概率分布的（上图中左边一支），而接下来要介绍的生成模型并不显式给出概率分布，而是直接在这个潜在概率分布上采样
#### Generative Adversarial Networks(GANs)
GAN 和 VAE 都从简单分布中采样 latent variable $z$，再通过生成网络生成样本 $x$。GAN 的 Generator 类似于 VAE 的 Decoder。区别是，VAE 通过显式的重建损失和 KL 损失来训练；GAN 则额外引入一个 Discriminator，让它判断真假，Generator 通过欺骗 Discriminator 来间接让生成分布 $p_G$ 接近真实数据分布 $p_{data}$
<div align="center">
  <img src="class_images/GANs_loss.jpg" width="500">
</div>

GANs的优势包括简约的损失函数和高质量的图片生成；但它的缺点包括没有合适的训练曲线参考，训练不稳定并且难以推广到大型模型和数据集
#### Diffusion Models
<div align="center">
  <img src="class_images/diffusion_model.jpg" width="500">
</div>

一种具体的 Diffusion Model 是 Rectified Flow。训练阶段，首先有一个$z$的简单分布，比如标准正态分布，然后有一个真实数据集。我们构建一个神经网络框架 $f_\theta$，然后从前述分布中随机采样一个$z$、$x$和$t$，其中$t$服从0到1的均匀分布，然后得到$x_t$，训练 $f_\theta$ 拟合$z-x$，损失函数如图。重复上述过程多次，得到一个 **分布层面** 的真实分布指向 $z$ 分布的神经网络（对应关系），注意这不是分布中每个具体点的对应关系
<div align="center">
  <img src="class_images/rectified_flow.jpg" width="500">
  <img src="class_images/rectified_flow_sample.jpg" width="500">
</div>
训练过程如右图所示。Rectified Flow的更多细节包括

* CFG 是一种控制生成结果的方法。训练时随机丢掉条件，让同一个模型同时学会有条件和无条件生成；采样时比较两者差异，并放大条件带来的方向，从而让生成结果更符合用户输入
* 不要简单均匀采样 t，而是更多采样中间噪声，甚至在高分辨率任务中偏向更高噪声
* 实践中最常用的框架如下图所示（Latent Diffusion Model）
* 关于Diffusion Model的详细介绍： https://sander.ai/2023/07/20/perspectives.html

<div align="center">
  <img src="class_images/LDMs.jpg" width="500">
</div>

In [ ]:
# rectified flow
import torch
import random
# training
for x in dataset:
    z = torch.randn_like(x)
    t = random.uniform(0, 1)
    xt = (1 - t) * x + t * z
    v = model(xt, t)
    loss = (v - (z - x)).pow(2).sum()

# sampling
sample = torch.randn(x_shape)
for t in torch.linspace(1, 0, steps=num_steps):
    v = model(sample, t)
    sample = sample - v / num_steps